# Constraints Tour

This notebook compares MASA's registered single-agent constraints on the same environment and action scripts. The point is not to train an agent; it is to see how the same labels can produce different safety metrics.

The matching static docs page is at `docs/Tutorials/Basics/Constraints Tour.md`.

## CPU-first setup

Keep the tutorial portable and quiet by selecting CPU before importing MASA/JAX modules.

In [ ]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

from pprint import pprint

ACTION_NAMES = {0: "left", 1: "right", 2: "down", 3: "up", 4: "stay"}


## Shared environment and scripts

All examples use `colour_grid_world`. The blue state is unsafe for cost-based constraints; the goal state is the reach target for reach-avoid.

In [ ]:
from masa.plugins.helpers import load_plugins
from masa.common.ltl import Atom, DFA
from masa.common.utils import make_env
from masa.envs.tabular.colour_grid_world import (
    BLUE_STATE,
    GOAL_STATE,
    GREEN_STATE,
    GRID_SIZE,
    PURPLE_STATE,
    START_STATE,
    cost_fn,
    label_fn,
)

load_plugins()

UNSAFE_SCRIPT = {"seed": 1, "actions": [2, 2, 2, 2]}
GOAL_SCRIPT = {"seed": 4, "actions": [2] * 8 + [1] * 8}

def make_never_blue_dfa():
    dfa = DFA([0, 1], 0, [1])
    dfa.add_edge(0, 1, Atom("blue"))
    return dfa


## Visual helpers

These helpers draw simple inline SVGs directly in the notebook. They avoid extra graphing dependencies and use rollout rows from the actual seeded environment runs.

In [ ]:
from html import escape

from IPython.display import SVG, display

CELL_SIZE = 42
PADDING = 34
SPECIAL_STATES = {
    START_STATE: ("start", "#dbeafe"),
    BLUE_STATE: ("blue", "#93c5fd"),
    GREEN_STATE: ("green", "#bbf7d0"),
    PURPLE_STATE: ("purple", "#ddd6fe"),
    GOAL_STATE: ("goal", "#dcfce7"),
}


def state_center(state):
    row, col = divmod(int(state), GRID_SIZE)
    return (
        PADDING + col * CELL_SIZE + CELL_SIZE / 2,
        PADDING + row * CELL_SIZE + CELL_SIZE / 2,
    )


def render_grid_trace_svg(rows, title):
    states = [int(row["obs"]) for row in rows]
    width = PADDING * 2 + GRID_SIZE * CELL_SIZE
    height = PADDING * 2 + GRID_SIZE * CELL_SIZE + 46
    points = " ".join(f"{x:.1f},{y:.1f}" for state in states for x, y in [state_center(state)])
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {width} {height}" width="{width}" height="{height}" role="img" aria-label="{escape(title)}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{PADDING}" y="24" font-family="sans-serif" font-size="16" font-weight="700" fill="#111827">{escape(title)}</text>',
    ]

    for state in range(GRID_SIZE * GRID_SIZE):
        row, col = divmod(state, GRID_SIZE)
        x = PADDING + col * CELL_SIZE
        y = PADDING + row * CELL_SIZE
        label, fill = SPECIAL_STATES.get(state, ("", "#f9fafb"))
        parts.append(f'<rect x="{x}" y="{y}" width="{CELL_SIZE}" height="{CELL_SIZE}" fill="{fill}" stroke="#d1d5db"/>')
        if label:
            parts.append(f'<text x="{x + 4}" y="{y + CELL_SIZE - 6}" font-family="sans-serif" font-size="9" fill="#374151">{label}</text>')

    if len(states) > 1:
        parts.append(f'<polyline points="{points}" fill="none" stroke="#111827" stroke-width="4" stroke-linejoin="round" stroke-linecap="round" opacity="0.72"/>')

    for step, state in enumerate(states):
        x, y = state_center(state)
        is_final = step == len(states) - 1
        fill = "#b91c1c" if state == BLUE_STATE else "#166534" if state == GOAL_STATE else "#111827"
        radius = 12 if is_final else 10
        parts.append(f'<circle cx="{x}" cy="{y}" r="{radius}" fill="{fill}" stroke="#ffffff" stroke-width="2"/>')
        parts.append(f'<text x="{x}" y="{y + 4}" text-anchor="middle" font-family="sans-serif" font-size="11" font-weight="700" fill="#ffffff">{step}</text>')

    legend_y = PADDING + GRID_SIZE * CELL_SIZE + 28
    parts.extend([
        f'<text x="{PADDING}" y="{legend_y}" font-family="sans-serif" font-size="12" fill="#374151">Numbers are reset/step indices from the actual seeded rollout.</text>',
        "</svg>",
    ])
    return "\n".join(parts)


def render_dfa_svg():
    return """
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 520 190" width="520" height="190" role="img" aria-label="Never blue DFA">
  <defs>
    <marker id="dfa-arrow" markerWidth="10" markerHeight="10" refX="9" refY="3" orient="auto" markerUnits="strokeWidth">
      <path d="M0,0 L0,6 L9,3 z" fill="#111827" />
    </marker>
  </defs>
  <rect width="100%" height="100%" fill="#ffffff"/>
  <text x="24" y="28" font-family="sans-serif" font-size="16" font-weight="700" fill="#111827">Never-blue DFA used by ltl_safety</text>
  <line x1="55" y1="96" x2="91" y2="96" stroke="#111827" stroke-width="2.5" marker-end="url(#dfa-arrow)"/>
  <circle cx="140" cy="96" r="40" fill="#dcfce7" stroke="#166534" stroke-width="3"/>
  <text x="140" y="91" text-anchor="middle" font-family="sans-serif" font-size="14" font-weight="700" fill="#14532d">q0</text>
  <text x="140" y="110" text-anchor="middle" font-family="sans-serif" font-size="11" fill="#14532d">safe</text>
  <circle cx="380" cy="96" r="42" fill="#fee2e2" stroke="#991b1b" stroke-width="3"/>
  <circle cx="380" cy="96" r="34" fill="none" stroke="#991b1b" stroke-width="2"/>
  <text x="380" y="91" text-anchor="middle" font-family="sans-serif" font-size="14" font-weight="700" fill="#7f1d1d">q1</text>
  <text x="380" y="110" text-anchor="middle" font-family="sans-serif" font-size="11" fill="#7f1d1d">unsafe</text>
  <path d="M180 96 C235 52 285 52 338 96" fill="none" stroke="#111827" stroke-width="2.5" marker-end="url(#dfa-arrow)"/>
  <text x="260" y="55" text-anchor="middle" font-family="sans-serif" font-size="13" font-weight="700" fill="#111827">blue</text>
  <path d="M122 58 C86 25 190 25 158 58" fill="none" stroke="#166534" stroke-width="2" marker-end="url(#dfa-arrow)"/>
  <text x="140" y="24" text-anchor="middle" font-family="sans-serif" font-size="11" fill="#166534">implicit not blue loop</text>
  <path d="M362 58 C326 25 430 25 398 58" fill="none" stroke="#991b1b" stroke-width="2" marker-end="url(#dfa-arrow)"/>
  <text x="380" y="24" text-anchor="middle" font-family="sans-serif" font-size="11" fill="#991b1b">implicit loop after violation</text>
</svg>
""".strip()


def render_constraint_semantics_svg():
    boxes = [
        ("cmdp", "sum cost <= budget"),
        ("prob", "unsafe fraction <= alpha"),
        ("pctl", "safety satisfied so far"),
        ("reach_avoid", "reach goal before blue"),
        ("ltl_safety", "DFA avoids accepting unsafe state"),
    ]
    width = 760
    height = 310
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {width} {height}" width="{width}" height="{height}" role="img" aria-label="Constraint semantics diagram">',
        '<defs><marker id="sem-arrow" markerWidth="10" markerHeight="10" refX="9" refY="3" orient="auto" markerUnits="strokeWidth"><path d="M0,0 L0,6 L9,3 z" fill="#4b5563" /></marker></defs>',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        '<text x="24" y="30" font-family="sans-serif" font-size="16" font-weight="700" fill="#111827">Same labels, different constraint semantics</text>',
        '<rect x="26" y="86" width="160" height="88" rx="8" fill="#f3f4f6" stroke="#9ca3af"/>',
        '<text x="106" y="117" text-anchor="middle" font-family="sans-serif" font-size="13" font-weight="700" fill="#111827">labels</text>',
        '<text x="106" y="140" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#374151">blue, goal, ...</text>',
    ]
    for index, (name, meaning) in enumerate(boxes):
        y = 58 + index * 46
        parts.append(f'<line x1="186" y1="130" x2="270" y2="{y + 20}" stroke="#4b5563" stroke-width="1.8" marker-end="url(#sem-arrow)"/>')
        parts.append(f'<rect x="280" y="{y}" width="430" height="36" rx="7" fill="#eff6ff" stroke="#93c5fd"/>')
        parts.append(f'<text x="300" y="{y + 23}" font-family="sans-serif" font-size="13" font-weight="700" fill="#1f2937">{escape(name)}</text>')
        parts.append(f'<text x="420" y="{y + 23}" font-family="sans-serif" font-size="12" fill="#374151">{escape(meaning)}</text>')
    parts.append("</svg>")
    return "\n".join(parts)


## Build each constraint

The base environment and labels stay fixed. Only the constraint wrapper and its configuration change.

In [ ]:
CONSTRAINT_NAMES = ["cmdp", "prob", "pctl", "reach_avoid", "ltl_safety"]

def build_constraint_env(name, max_episode_steps=40):
    if name == "cmdp":
        return make_env(
            "colour_grid_world",
            "cmdp",
            max_episode_steps,
            label_fn=label_fn,
            cost_fn=cost_fn,
            budget=0.0,
        )
    if name == "prob":
        return make_env(
            "colour_grid_world",
            "prob",
            max_episode_steps,
            label_fn=label_fn,
            cost_fn=cost_fn,
            alpha=0.1,
        )
    if name == "pctl":
        return make_env(
            "colour_grid_world",
            "pctl",
            max_episode_steps,
            label_fn=label_fn,
            cost_fn=cost_fn,
            alpha=0.01,
        )
    if name == "reach_avoid":
        return make_env(
            "colour_grid_world",
            "reach_avoid",
            max_episode_steps,
            label_fn=label_fn,
            avoid_label="blue",
            reach_label="goal",
        )
    if name == "ltl_safety":
        return make_env(
            "colour_grid_world",
            "ltl_safety",
            max_episode_steps,
            label_fn=label_fn,
            dfa=make_never_blue_dfa(),
            obs_type="dict",
        )
    raise ValueError(f"unknown constraint: {name}")


## Constraint diagrams

The DFA diagram is the property used by `ltl_safety`. The semantics diagram is the core lesson of this tutorial: the labels are shared, but each constraint interprets them differently.

In [ ]:
display(SVG(render_dfa_svg()))
display(SVG(render_constraint_semantics_svg()))


## Rollout helper

The helper records a compact final row for each constraint. `ltl_safety` augments observations with automaton state, so the observation simplifier handles dictionaries as well as integers.

In [ ]:
def simplify_obs(obs):
    if isinstance(obs, dict):
        return {key: simplify_obs(value) for key, value in obs.items()}
    try:
        return int(obs)
    except (TypeError, ValueError):
        return obs

def run_constraint(name, *, seed, actions, max_episode_steps=40):
    env = build_constraint_env(name, max_episode_steps=max_episode_steps)
    obs, info = env.reset(seed=seed)
    rows = [
        {
            "event": "reset",
            "obs": simplify_obs(obs),
            "labels": sorted(info["labels"]),
            "constraint": info["constraint"],
            "automaton_state": info.get("automaton_state"),
        }
    ]

    for step, action in enumerate(actions, start=1):
        obs, reward, terminated, truncated, info = env.step(action)
        rows.append(
            {
                "event": f"step_{step}",
                "action": ACTION_NAMES[action],
                "obs": simplify_obs(obs),
                "reward": float(reward),
                "terminated": bool(terminated),
                "truncated": bool(truncated),
                "labels": sorted(info["labels"]),
                "constraint": info["constraint"],
                "automaton_state": info.get("automaton_state"),
            }
        )
        if terminated or truncated:
            break

    env.close()
    return rows

def final_metrics_for(script):
    return {
        name: run_constraint(name, seed=script["seed"], actions=script["actions"])[-1]
        for name in CONSTRAINT_NAMES
    }


## Unsafe script: reach blue

With seed `1`, actions `[2, 2, 2, 2]` reach the blue state. Compare how each constraint reports the same labelled event.

In [ ]:
unsafe_trace = run_constraint("cmdp", seed=UNSAFE_SCRIPT["seed"], actions=UNSAFE_SCRIPT["actions"])
display(SVG(render_grid_trace_svg(unsafe_trace, "Unsafe script: blue is reached")))

unsafe_results = final_metrics_for(UNSAFE_SCRIPT)
pprint(unsafe_results)

assert unsafe_trace[-1]["obs"] == BLUE_STATE
assert unsafe_results["cmdp"]["constraint"]["step"]["cost"] == 1.0
assert unsafe_results["prob"]["constraint"]["episode"]["satisfied"] == 0.0
assert unsafe_results["pctl"]["constraint"]["episode"]["satisfied"] == 0.0
assert unsafe_results["reach_avoid"]["constraint"]["episode"]["violated"] is True
assert unsafe_results["ltl_safety"]["constraint"]["episode"]["satisfied"] == 0.0


## Goal script: reach goal without blue

With seed `4`, actions `[2] * 8 + [1] * 8` reach the goal state without visiting blue. Reach-avoid should be satisfied, and the never-blue LTL monitor should remain safe.

In [ ]:
goal_trace = run_constraint("cmdp", seed=GOAL_SCRIPT["seed"], actions=GOAL_SCRIPT["actions"])
display(SVG(render_grid_trace_svg(goal_trace, "Goal script: goal is reached without blue")))

goal_results = final_metrics_for(GOAL_SCRIPT)
pprint(goal_results)

assert goal_trace[-1]["obs"] == GOAL_STATE
assert goal_results["cmdp"]["constraint"]["episode"]["satisfied"] == 1.0
assert goal_results["prob"]["constraint"]["episode"]["satisfied"] == 1.0
assert goal_results["pctl"]["constraint"]["episode"]["satisfied"] == 1.0
assert goal_results["reach_avoid"]["constraint"]["episode"]["satisfied"] == 1.0
assert goal_results["ltl_safety"]["constraint"]["episode"]["satisfied"] == 1.0
assert goal_results["reach_avoid"]["labels"] == ["goal"]


## How to read the differences

- `cmdp` accumulates scalar cost and checks it against a budget.
- `prob` tracks the empirical fraction of unsafe observations and checks it against `alpha`.
- `pctl` currently behaves as a safety-so-far tracker over unsafe events.
- `reach_avoid` separately tracks whether the target was reached and whether the avoid label was ever seen.
- `ltl_safety` advances a DFA and reports violations when the automaton enters an accepting unsafe state.

The raw environment labels are the same. The constraint determines how those labels become safety state and metrics.